# Lab 14: Traces, latency and cost
**Module 7.2: Monitoring and cost.**  **Time:** 25 minutes.

**Goal:** know where time and money go in an AI request, and forecast the monthly bill before launch.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import time, contextlib, pandas as pd
qc, _ = lk.build_policy_index()
lk.reset_calls()

## 1. A trace: one request, broken into timed steps (spans)
Tools such as OpenTelemetry, Langfuse or your cloud's tracing service record exactly this, automatically.

In [ ]:
TRACE = []

@contextlib.contextmanager
def span(name, **attrs):
    t0 = time.perf_counter()
    yield
    TRACE.append({"span": name, "ms": round((time.perf_counter() - t0) * 1000, 1), **attrs})

def traced_answer(question):
    with span("embed+search"):
        hits = lk.search(qc, question, k=4)
    with span("build prompt", passages=len(hits)):
        prompt = f"Context:\n{lk.format_context(hits)}\n\nQuestion: {question}"
    with span("llm call", model=lk.model_name("big")):
        answer = lk.chat(prompt, system=lk.RAG_SYSTEM, tag="traced")
    return answer

TRACE.clear()
print(traced_answer("What is the meal allowance for international travel?"))
pd.DataFrame(TRACE)

Which step dominates? In most RAG systems the LLM call is most of the latency **and** all of the variable cost.

## 2. A small load test: 8 questions

In [ ]:
questions = ["How many sick leave days do I get?", "What is the hotel limit in Chennai?",
             "How do I report a phishing email?", "What is the gift limit from a vendor?",
             "When are expense claims due?", "How long are CCTV recordings kept?",
             "What is the chat first response target?", "How often are laptops replaced?"]
for q in questions:
    lk.rag_answer(q, qc, tag="load")
calls = lk.calls_df()
calls[calls.tag == "load"][["model", "latency_s", "prompt_tokens", "completion_tokens", "est_cost_usd"]]

In [ ]:
load = calls[calls.tag == "load"]
summary = {
    "requests": len(load),
    "latency p50 (s)": load.latency_s.median(),
    "latency p95 (s)": load.latency_s.quantile(0.95),
    "avg input tokens": round(load.prompt_tokens.mean()),
    "avg output tokens": round(load.completion_tokens.mean()),
    "avg cost per request (USD)": round(load.est_cost_usd.mean(), 6),
}
summary

In [ ]:
ax = load.latency_s.plot.bar(title="Latency per request (s)", figsize=(7, 3))

Input tokens are much larger than output tokens. Why? (The retrieved passages.) That is your main cost lever.

## 3. Forecast the monthly bill
Edit the numbers for a real rollout.

In [ ]:
employees = 5000
questions_per_employee_per_day = 3
working_days = 22
usd_to_inr = 88

per_req = summary["avg cost per request (USD)"]
monthly_requests = employees * questions_per_employee_per_day * working_days
monthly_usd = monthly_requests * per_req
print(f"{monthly_requests:,} requests/month  ->  about USD {monthly_usd:,.2f}  (INR {monthly_usd*usd_to_inr:,.0f})")

## 4. Budget alert
A simple guard that a platform team would run per tenant or per application.

In [ ]:
DAILY_BUDGET_USD = 5.0
def check_budget(df, budget=DAILY_BUDGET_USD):
    spent = df.est_cost_usd.sum()
    pct = spent / budget
    level = "OK" if pct < 0.8 else ("WARN" if pct < 1 else "STOP")
    print(f"spent today USD {spent:.4f} of {budget} ({pct:.1%})  ->  {level}")
    return level

check_budget(calls)

## Try this
1. Halve `k` (passages) in `rag_answer(..., k=2)` for the load test. What happens to tokens, cost and answer quality?
2. Switch the load test to the small model. Recompute the forecast.

**Discuss:** who pays? Showback or chargeback per business unit needs the `tag` column to carry a tenant id.